In [1]:
import os
import time
import pandas as pd
import requests
import wget
from requests.adapters import HTTPAdapter, Retry
from joblib import Parallel, delayed

In [2]:
# Fastq folder
os.makedirs("fastq", exist_ok=True)

# Load your metadata Excel file
df = pd.read_excel("metadata.xlsx")

In [3]:
# Use a persistent session with retries for ENA API calls
session = requests.Session()
retries = Retry(total=5, backoff_factor=0.3)
session.mount("https://", HTTPAdapter(max_retries=retries))

def get_fastq_links(run_accession):
    """Query ENA API for fastq_ftp links for a given run accession."""
    url = (
        "https://www.ebi.ac.uk/ena/portal/api/filereport"
        f"?accession={run_accession}&result=read_run&fields=fastq_ftp&format=tsv"
    )
    try:
        r = session.get(url, timeout=10)
        if r.status_code != 200:
            print(f"⚠️ ENA API error for {run_accession}")
            return []
        lines = r.text.strip().split("\n")
        if len(lines) > 1 and lines[1].strip():
            fastq_field = lines[1].split("\t")[-1]
            return fastq_field.split(";")
        return []
    except Exception as e:
        print(f"⚠️ Error fetching links for {run_accession}: {e}")
        return []

p.s. For the next iteration, make sure that case 1 is to use the ENA accession

In [ ]:
download_tasks = []

for idx, row in df.iterrows():
    ena_run = str(row["ena_run"]).strip()
    ena_experiment = str(row["ena_experiment"]).strip()
    ena_sample = str(row["ena_sample"]).strip()
    aux_link = str(row.get("auxillary.ftp.link")).strip()

    # Case 1: Use auxiliary link
    if pd.notnull(aux_link) and aux_link.lower() != "nan":
        link = aux_link if aux_link.startswith("ftp://") else f"ftp://{aux_link}"
        output_name = link.split("/")[-1]  # Just the filename from FTP
        download_tasks.append((link, output_name))

    else:
        # Case 2: Use ENA accession
        accession = None
        if pd.notnull(ena_run) and ena_run.startswith(("ERR", "SRR", "SAM", "ERX")):
            accession = ena_run
        elif pd.notnull(ena_experiment) and ena_experiment.startswith(("ERX", "ERS")):
            accession = ena_experiment
        elif pd.notnull(ena_sample) and ena_sample.startswith(("ERS", "SAM")):
            accession = ena_sample

        if accession:
            links = get_fastq_links(accession)
            time.sleep(0.2)
            if links:
                for link in links:
                    link = link if link.startswith("ftp://") else f"ftp://{link}"
                    output_name = link.split("/")[-1] 
                    download_tasks.append((link, output_name))
            else:
                print(f"⚠️ No FASTQ links found for {accession}")
        else:
            print(f"⚠️ Invalid accession in row {idx}")

print(f"📦 Prepared {len(download_tasks)} download tasks.")
# 4m 30.4s 
# 5m 2.7s

📦 Prepared 1242 download tasks.


In [7]:
max_retries = 3
wait_seconds = 5

def download_file(url, output_path):
    filename = os.path.basename(output_path)
    final_path = os.path.join("fastq", filename)

    # Check if file already exists
    if os.path.exists(final_path):
        print(f"⏭️ Skipped (already exists): {filename}")
        return f"Skipped: {filename}"

    # Retry logic
    for attempt in range(1, max_retries + 1):
        try:
            print(f"⬇️ Attempt {attempt}/{max_retries}: {url} -> {filename}")
            wget.download(url, out=final_path)
            print(f"\n✅ Downloaded: {filename}")
            return f"Downloaded: {filename}"
        except Exception as e:
            print(f"❌ Failed attempt {attempt}: {e}")
            if attempt < max_retries:
                print(f"🔁 Retrying in {wait_seconds} seconds...")
                time.sleep(wait_seconds)
            else:
                print(f"❌ Final failure after {max_retries} attempts.")
                return f"Failed: {filename} | Error: {e}"

p.s. Make sure that there is a log file where it stores the results of the downloads instead

p.s. Also add tdqm so we know how many files have been processed already (or how much of it in %)

In [8]:
# Perform parallel downloads
Parallel(n_jobs=8)(
    delayed(download_file)(url, output) for url, output in download_tasks
)

⬇️ Attempt 1/3: ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR251/007/ERR2513827/ERR2513827_2.fastq.gz -> ERR2513827_2.fastq.gz
⬇️ Attempt 1/3: ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR251/007/ERR2513827/ERR2513827_1.fastq.gz -> ERR2513827_1.fastq.gz
⬇️ Attempt 1/3: ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR481/009/ERR4814389/ERR4814389_1.fastq.gz -> ERR4814389_1.fastq.gz
⬇️ Attempt 1/3: ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR481/009/ERR4814389/ERR4814389_2.fastq.gz -> ERR4814389_2.fastq.gz
⬇️ Attempt 1/3: ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR552/ERR552394/ERR552394_1.fastq.gz -> ERR552394_1.fastq.gz
⬇️ Attempt 1/3: ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR552/ERR552394/ERR552394_2.fastq.gz -> ERR552394_2.fastq.gz
⬇️ Attempt 1/3: ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR479/000/ERR4798220/ERR4798220_2.fastq.gz -> ERR4798220_2.fastq.gz⬇️ Attempt 1/3: ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR479/000/ERR4798220/ERR4798220_1.fastq.gz -> ERR4798220_1.fastq.gz


✅ Downloaded: ERR2513827_2.fastq.gz
⬇️ Attempt 1/3: f

KeyboardInterrupt: 